In [1]:
from google.colab import drive
drive.mount('/content/drive', force_remount=True)
!ln -s /content/drive/MyDrive/Colab\ Notebooks/SES-Inference /mydrive

Mounted at /content/drive


In [2]:
%cd /mydrive/scripts/
!ls 

/content/drive/MyDrive/Colab Notebooks/SES-Inference/scripts
batch_cells.py	      batch_staticmaps.py
batch_cells.sh	      batch_staticmaps.sh
batch_iwi.py	      batch_train_by_country_scratch.sh
batch_iwi.sh	      batch_xgboost_model.sh
batch_locations.py    cnn_augmentation.py
batch_locations.sh    cnn_performance.py
batch_marketing.py    cnn_predict.py
batch_marketing.sh    cnn_train.py
batch_movement.py     get_VOC_data.sh
batch_movement.sh     get_xview_data_utilities.sh
batch_nightlights.py  prepare.sh
batch_nightlights.sh  __pycache__
batch_osm.py	      survey_decompress.sh
batch_osm.sh	      test_hyperopt.py
batch_population.py   xgboost_performance.py
batch_population.sh   xgboost_predict.py
batch_pplaces.py      xgboost_predict.sh
batch_pplaces.sh      xgboost_train.py
batch_sampling.py     xgboost_train.sh
batch_sampling.sh


# Pre-requisites


1.   Surveys + GPS data: [DHS](https://dhsprogram.com/data/available-datasets.cfm) (\*) (**)
```
data/<country>/survey/DHS/<year>/
```

2.   Cell towers: [OpenCelliD](https://www.opencellid.org/downloads.php]) (\*)

    *Make sure the original file is splitted into countries (see _OpenCellid.ipynb)*
```
data/OpenCellid/
```

3.   High resolution population maps: [Facebook & Michigan](https://data.humdata.org/dataset/highresolutionpopulationdensitymaps)
```
data/<country>/population/Facebook/
```

4.   Movement data: [Facebook](https://research.fb.com/blog/2017/06/facebook-disaster-maps-methodology/) (**)
```
data/<country>/movement/Facebook/
```

5.   Get API keys
     
     5.1.   Geopy (user_agent: just a random string)
     ```
     data/maps/geopy/user_agent
     ```
     5.2    [Google Maps Static API](https://developers.google.com/earth-engine/guides/app_key)
     ```
     data/maps/staticmaps/api_key
     data/maps/staticmaps/secret
     ```
     5.3    [Google Earth Engine]()
     ```
     data/maps/googleearth/api_key
     data/maps/googleearth/project_id
     ```
     5.4    [Facebook Marketing API](https://developers.facebook.com/docs/marketing-apis/#marketing-api)
     ```
     data/Facebook/MarketingAPI/tokens/tokens<ID>.json
     ```


(\*) need authentication
(**) needs application / approval

# Initialization

In [3]:
!chmod +x *.sh 

In [4]:
available_data = {'Sierra Leone': {'years':'2016,2019', 'code':'SL'}}

In [5]:
country = "Sierra Leone"

In [ ]:
# prepare folder structure per country and install library dependencies
!./prepare.sh "$country"

In [8]:
# country and years
root = "../data/" + country
years = available_data[country]['years']
code = available_data[country]['code']
print(root, code, years)

../data/Sierra Leone SL 2016,2019


# 1. Feature extraction

* Load survey
* Compute IWI per household
* Compute mean IWI per cluster
* Cellular antennas (OpenCelliD)
* Population (Facebook & Uni Michigan)
* Nightlight intensity (Google Earth Engine)
* Roads, Junctions, Buildings, POIs (OpenStreetMap)
* Movement of people / between tiles - before corona crisis (Facebook)
* Satellite day-light images (Google Maps Statis API)
* Feature Maps (extracted from CNN)

## 1.1. DHS clusters

In [ ]:
# 1. IWI
njobs = 10
!./batch_iwi.sh -r "$root" -c $code -y $years -n $njobs

2021-11-25 19:17:27.083065: E tensorflow/stream_executor/cuda/cuda_driver.cc:271] failed call to cuInit: CUDA_ERROR_NO_DEVICE: no CUDA-capable device is detected
r: ../data/Uganda
c: UG
y: 2016,2018
n: 10

1. Loading data...
100% 2/2 [00:16<00:00,  8.49s/it]
100% 2/2 [00:00<00:00, 14.56it/s]
   year  ... hv216
0  2016  ...     2
1  2016  ...     1
2  2016  ...     1
3  2016  ...     4
4  2016  ...     1

[5 rows x 24 columns]
- shape survey: (27798, 24)
- shape cluster: (1001, 22)
- years survey: year
2016    19447
2018     8351
dtype: int64
- years cluster: DHSYEAR
2016    685
2018    316
dtype: int64

2. Computing IWI per household and cluster...
27798it [00:11, 2462.35it/s]
   year  ...        iwi
0  2016  ...  78.433205
1  2016  ...  56.689369
2  2016  ...  40.020046
3  2016  ...  62.008667
4  2016  ...  41.727264

[5 rows x 25 columns]
            DHSID  ...    std_iwi
0  UG201600000001  ...  12.820034
1  UG201600000002  ...  17.185404
2  UG201600000003  ...  15.608082
3  UG201600

In [ ]:
# 2. Cells
distance = 10.0
!./batch_cells.sh -r "$root" -y $years -d $distance
# --- 121.5247437953949 seconds ---

2021-12-07 10:29:07.151470: E tensorflow/stream_executor/cuda/cuda_driver.cc:271] failed call to cuInit: CUDA_ERROR_NO_DEVICE: no CUDA-capable device is detected
r: ../data/Sierra Leone
y: 2016,2019
d: 10.0
../data/Sierra Leone/connectivity/cell_towers_Sierra Leone.csv
100% 893/893 [01:51<00:00,  8.01it/s]
            DHSID  ...  towers_in_10.0km
0  SL201600000001  ...               0.0
1  SL201600000002  ...               0.0
2  SL201600000003  ...               0.0
3  SL201600000004  ...               0.0
4  SL201600000005  ...               0.0

[5 rows x 10 columns]
(893, 10)
../data/Sierra Leone/results/features/clusters/DHS2019_MIS2016_iwi_cluster_cells.csv saved!
--- 112.42118644714355 seconds ---


In [ ]:
# 3. population
!./batch_population.sh -r "$root" -y $years
# --- 61.78075695037842 seconds ---

2021-12-07 10:31:05.665953: E tensorflow/stream_executor/cuda/cuda_driver.cc:271] failed call to cuInit: CUDA_ERROR_NO_DEVICE: no CUDA-capable device is detected
r: ../data/Sierra Leone
y: 2016,2019
100% 4/4 [00:07<00:00,  1.84s/it]
            DHSID  ...  population_in_10.0km
0  SL201600000001  ...           4708.640859
1  SL201600000002  ...           8602.876948
2  SL201600000003  ...          12150.389127
3  SL201600000004  ...          17913.836826
4  SL201600000005  ...          21867.159616

[5 rows x 10 columns]
(893, 10)
../data/Sierra Leone/results/features/clusters/DHS2019_MIS2016_iwi_cluster_population.csv saved!
--- 59.4845027923584 seconds ---


In [ ]:
# 4. nightlights (maybe pass rad gte threshold)
!./batch_nightlights.sh -r "$root" -y $years
# --- 29.69801950454712 seconds ---

2021-12-06 22:47:37.411404: E tensorflow/stream_executor/cuda/cuda_driver.cc:271] failed call to cuInit: CUDA_ERROR_NO_DEVICE: no CUDA-capable device is detected
r: ../data/Sierra Leone
y: 2016,2019
a: 
p: 

  0% 0/8 [00:00<?, ?it/s]To authorize access needed by Earth Engine, open the following URL in a web browser and follow the instructions. If the web browser does not start automatically, please manually browse the URL below.

    https://accounts.google.com/o/oauth2/auth?client_id=517222506229-vsmmajv00ul0bs7p89v5m89qs8eb9359.apps.googleusercontent.com&scope=https%3A%2F%2Fwww.googleapis.com%2Fauth%2Fearthengine+https%3A%2F%2Fwww.googleapis.com%2Fauth%2Fdevstorage.full_control&redirect_uri=urn%3Aietf%3Awg%3Aoauth%3A2.0%3Aoob&response_type=code&code_challenge=2i3-CssVbkUWHEPYjai8FKW_1_wJ3B50HJUSIMTQp8M&code_challenge_method=S256

The authorization workflow will generate a code, which you should paste in the box below. 
Enter verification code: 4/1AX4XfWjPIPV7peLjfq5M6BVxtpVy5G0R6jtG_

In [ ]:
# 5. OpenStreetMap
!./batch_osm.sh -r "$root" -y $years
# --- 17544.289302825928 seconds ---
# --- 21451.835978269577 seconds ---
# --- 2028.5190045833588 seconds ---
# --- 279.0613946914673 seconds ---

2021-12-06 22:35:14.414490: E tensorflow/stream_executor/cuda/cuda_driver.cc:271] failed call to cuInit: CUDA_ERROR_NO_DEVICE: no CUDA-capable device is detected
r: ../data/Sierra Leone
y: 2016,2019
100% 893/893 [04:36<00:00,  3.23it/s]
            DHSID  ... townhall_dist
0  SL201600000001  ...             0
1  SL201600000002  ...             0
2  SL201600000003  ...             0
3  SL201600000004  ...             0
4  SL201600000005  ...             0

[5 rows x 55 columns]
(893, 55)
../data/Sierra Leone/results/features/clusters/DHS2019_MIS2016_iwi_cluster_OSM.csv saved!
--- 279.0613946914673 seconds ---


In [ ]:
!ls /mydrive/data/Sierra\ Leone/cache/FBM | wc -lc

  34828 2383684


In [ ]:
# 6. Facebook marketing
tokensdir = '/mydrive/data/Facebook/MarketingAPI/tokens'

!./batch_marketing.sh -r "$root" -y $years -t "$tokensdir"
# --- 23246.720271110535 seconds ---
# --- 9722.878120183945 seconds ---

2021-12-07 07:45:36.266001: E tensorflow/stream_executor/cuda/cuda_driver.cc:271] failed call to cuInit: CUDA_ERROR_NO_DEVICE: no CUDA-capable device is detected
r: ../data/Sierra Leone
y: 2016,2019
t: /mydrive/data/Facebook/MarketingAPI/tokens
100% 893/893 [2:42:02<00:00, 10.89s/it]
../data/Sierra Leone/results/features/clusters/DHS2019_MIS2016_iwi_cluster_FBM.csv saved!
--- 9722.878120183945 seconds ---


In [ ]:
# 7. Facebook Movement between tile (before Corona crisis)
njobs = 10
!./batch_movement.sh -r "$root" -y $years -j $njobs
# --- 118.12501120567322 seconds ---

2021-12-07 07:42:18.128170: E tensorflow/stream_executor/cuda/cuda_driver.cc:271] failed call to cuInit: CUDA_ERROR_NO_DEVICE: no CUDA-capable device is detected
r: ../data/Sierra Leone
y: 2016,2019
j: 10
(488, 488) (488, 488) (488, 488)
2617 2617.0
235313.95
0.0 293.47756008784086 0.2833244866020458
../data/Sierra Leone/results/features/clusters/DHS2019_MIS2016_iwi_cluster_FBMV.csv saved!
--- 118.12501120567322 seconds ---


## 1.2. Populated places

In [ ]:
# 1. OpenStreetMap Populated places
!./batch_pplaces.sh -r "$root"
# --- 44.763751745224 seconds ---

2021-12-07 11:52:25.780080: E tensorflow/stream_executor/cuda/cuda_driver.cc:271] failed call to cuInit: CUDA_ERROR_NO_DEVICE: no CUDA-capable device is detected
r: ../data/Sierra Leone
100% 9568/9568 [00:43<00:00, 220.81it/s]
../data/Sierra Leone/results/features/pplaces/PPLACES.csv saved!
(9568, 12)
--- 44.763751745224 seconds ---


In [ ]:
# 2. OpenStreetMap Features
!./batch_osm.sh -r "$root"
# --- 70594.4 seconds --- SL (~19 hours, 10K places)
# --- 4885.949688911438 seconds ---

2021-12-08 15:36:09.835874: E tensorflow/stream_executor/cuda/cuda_driver.cc:271] failed call to cuInit: CUDA_ERROR_NO_DEVICE: no CUDA-capable device is detected
r: ../data/Sierra Leone
y: 
100% 9568/9568 [1:21:23<00:00,  1.96it/s]
         OSMID  ...  townhall_dist
0   27565056.0  ...              0
1  314001434.0  ...        277.369
2  314005602.0  ...              0
3  314007819.0  ...              0
4  320058940.0  ...              0

[5 rows x 55 columns]
(9568, 55)
../data/Sierra Leone/results/features/pplaces/PPLACES_OSM.csv saved!
--- 4885.949688911438 seconds ---


In [ ]:
# 3. Cells
distance = 10.0
!./batch_cells.sh -r "$root" -d $distance
# --- 1282.0412645339966 seconds ---

2021-12-08 01:01:52.180803: E tensorflow/stream_executor/cuda/cuda_driver.cc:271] failed call to cuInit: CUDA_ERROR_NO_DEVICE: no CUDA-capable device is detected
r: ../data/Sierra Leone
y: 
d: 10.0
../data/Sierra Leone/connectivity/cell_towers_Sierra Leone.csv
100% 9568/9568 [21:20<00:00,  7.47it/s]
       OSMID  ...  towers_in_10.0km
0   27565056  ...             549.0
1  314001434  ...              86.0
2  314005602  ...              83.0
3  314007819  ...              11.0
4  320058940  ...              14.0

[5 rows x 10 columns]
(9568, 10)
../data/Sierra Leone/results/features/pplaces/PPLACES_cells.csv saved!
--- 1282.0412645339966 seconds ---


In [ ]:
# 4. population
!./batch_population.sh -r "$root"
# --- 107.97806024551392 seconds ---

2021-12-08 01:25:17.824076: E tensorflow/stream_executor/cuda/cuda_driver.cc:271] failed call to cuInit: CUDA_ERROR_NO_DEVICE: no CUDA-capable device is detected
r: ../data/Sierra Leone
y: 
100% 4/4 [00:53<00:00, 13.39s/it]
       OSMID  ...  population_in_10.0km
0   27565056  ...         665944.121775
1  314001434  ...         322010.257744
2  314005602  ...         229062.240826
3  314007819  ...         100964.134167
4  320058940  ...          44350.093036

[5 rows x 10 columns]
(9568, 10)
../data/Sierra Leone/results/features/pplaces/PPLACES_population.csv saved!
--- 107.97806024551392 seconds ---


In [ ]:
# 5. nightlights
!earthengine authenticate --quiet
!earthengine authenticate --code-verifier=GjqXno1WQVDHvPm4eY5mdbu86RA9A8Il-PhjdXKc3c0 --authorization-code=4/1AX4XfWjfsErMGSxCUeqrcKs39uaXqBD-fjTI-yIQNVXGEWDeFrn9P7FrBXI

key = "../data/maps/googleearth/api_key"
project = "../data/maps/googleearth/project_id"
!./batch_nightlights.sh -r "$root" -a "$key" -p "$project" 
# --- 4157.716005325317 seconds ---
# --- 8.377232789993286 seconds ---

2021-12-08 15:08:44.106923: E tensorflow/stream_executor/cuda/cuda_driver.cc:271] failed call to cuInit: CUDA_ERROR_NO_DEVICE: no CUDA-capable device is detected
r: ../data/Sierra Leone
y: 
a: ../data/maps/googleearth/api_key
p: ../data/maps/googleearth/project_id

  0% 0/4 [00:00<?, ?it/s]../data/Sierra Leone/cache/VIIRSPP/size9568_bmeter1609.34_radthres10.0_scale30_b1_2.csv loaded!
../data/Sierra Leone/cache/VIIRSPP/size9568_bmeter1609.34_radthres10.0_scale30_b2_2.csv loaded!
 25% 1/4 [00:02<00:07,  2.53s/it]../data/Sierra Leone/cache/VIIRSPP/size9568_bmeter2000_radthres10.0_scale30_b1_2.csv loaded!
../data/Sierra Leone/cache/VIIRSPP/size9568_bmeter2000_radthres10.0_scale30_b2_2.csv loaded!
 50% 2/4 [00:03<00:03,  1.81s/it]../data/Sierra Leone/cache/VIIRSPP/size9568_bmeter5000_radthres10.0_scale30_b1_2.csv loaded!
../data/Sierra Leone/cache/VIIRSPP/size9568_bmeter5000_radthres10.0_scale30_b2_2.csv loaded!
 75% 3/4 [00:04<00:01,  1.51s/it]../data/Sierra Leone/cache/VIIRSPP/size9568_bm

In [ ]:
# 6. Facebook marketing
tokensdir = '../data/Facebook/MarketingAPI/tokens'
!./batch_marketing.sh -r "$root" -t "$tokensdir"
# --- 61284.84282016754 seconds --- UG

In [ ]:
# 7. Facebook Movement between tile (Corona crisis)
njobs = 20
!./batch_movement.sh -r "$root" -j $njobs
# --- 157.10024285316467 seconds ---

2021-12-08 17:10:41.545753: E tensorflow/stream_executor/cuda/cuda_driver.cc:271] failed call to cuInit: CUDA_ERROR_NO_DEVICE: no CUDA-capable device is detected
r: ../data/Sierra Leone
y: 
j: 20
(488, 488) (488, 488) (488, 488)
2617 2617.0
235313.95
0.0 293.47756008784086 0.2833244866020458
../data/Sierra Leone/results/features/pplaces/PPLACES_FBMV.csv saved!
--- 157.10024285316467 seconds ---


# 2. Pre-processing

## 2.1 Change DHS cluster location

In [ ]:
# 1. Change clusters fot populated places: cc, ccur, gc, gcur, rc, none
for dhsloc in ['rc']: #['none', 'cc', 'ccur', 'gc', 'gcur', 'rc']:
  !./batch_locations.sh -r "$root" -y $years -o $dhsloc

2021-12-08 22:33:28.122227: E tensorflow/stream_executor/cuda/cuda_driver.cc:271] failed call to cuInit: CUDA_ERROR_NO_DEVICE: no CUDA-capable device is detected
r: ../data/Sierra Leone
y: 2016,2019
o: rc
clusters (893, 28)
pplaces (9568, 14)
rc: Urban no change, Rural to closest rural PPlace progressively


0:URBAN,2000
year:2019
cluster:  (893, 28) (209, 28)
results (clusters) (209, 9)
year:2016
cluster:  (893, 28) (99, 28)
results (clusters) (308, 9)


1:RURAL,5000
year:2019
cluster:  (893, 28) (348, 28)
pplaces:  (9568, 14) (9504, 14)
results (clusters+pplaces) (656, 9)
year:2016
cluster:  (893, 28) (237, 28)
pplaces:  (9568, 14) (9504, 14)
results (clusters+pplaces) (893, 9)
df_results (893, 9)
dhs_notchanged None
df_results (893, 9)
within (893, 9)
cluster: (893, 28)
pplaces: (9568, 14)
results (match): (893, 9)
valid: (893, 9)
ignored: 0 0.0
rural
0    308
1    585
dtype: int64
final: (893, 9) 893 585 893
dhs_rural
0    308
1    585
dtype: int64
clusters:  586 pplaces:  8983
emp

## 2.2 Sampling

In [ ]:
dhsloc = 'rc'
kfolds = 5
repeats = 5
seeds = None
traintype = 'all' #all, oldest, newest
# NOTE: if years is one year (e.g., 2016) then traintype must be 'all' (exception is raised otherwise)

!./batch_sampling.sh -r "$root" -y "$years" -o "$dhsloc" -t "$traintype" -k $kfolds -e $repeats -s "$seeds"

2021-12-09 01:13:36.148778: E tensorflow/stream_executor/cuda/cuda_driver.cc:271] failed call to cuInit: CUDA_ERROR_NO_DEVICE: no CUDA-capable device is detected
r: ../data/Sierra Leone
y: 2016,2019
o: rc
t: all
k: 5
e: 5
s: None
893 records loaded.
893 records in ['2016', '2019'].
random_state: 4137318063
records run #1 893
1.0    183
Name: test, dtype: int64
train    532
val      178
Name: fold1, dtype: int64
train    532
val      178
Name: fold2, dtype: int64
train    533
val      177
Name: fold3, dtype: int64
train    533
val      177
Name: fold4, dtype: int64
../data/Sierra Leone/results/samples/DHS2019_MIS2016_all_rc/epoch1-rs4137318063/data.csv saved!
random_state: 528327992
records run #2 893
1.0    183
Name: test, dtype: int64
train    532
val      178
Name: fold1, dtype: int64
train    532
val      178
Name: fold2, dtype: int64
train    533
val      177
Name: fold3, dtype: int64
train    533
val      177
Name: fold4, dtype: int64
../data/Sierra Leone/results/samples/DHS2019_M

# 3. XGBoost

## 3.1. Training (DHS)

In [ ]:
# 2. Training XGBoost model (feature-based + CNN model)
traintype = 'all'
dhsloc = 'rc'
isregression = True
viirsnorm = True

print(f'Running: {path}...')

params = ["-r '{}'".format(root),
        "-y {}".format(years),
        "-d '{}'".format(dhsloc),
        "-t {}".format(traintype),
        "-r {}".format(isregression), 
        "-v {}".format(viirsnorm)]

command = "./xgb_train.sh {}".format(' '.join(params))
print(command)
!$command



---

